## 1. Data Loading and Initial Setup

In this section, we import the required Python libraries and load the four CairoGo datasets: trips, customers, drivers, and zones.

The `request_ts`, `signup_date`, and `join_date` columns are parsed as datetime values to support temporal analysis during the data leakage investigation.

We also verify that the dataset folder exists and inspect the shape of each dataset to ensure successful loading.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries loaded ✓")

# Dataset folder
PACK = Path("../data/raw/cairogo_pack").resolve()

if not PACK.exists():
    raise FileNotFoundError(f"Dataset folder not found: {PACK}")

# Load datasets
trips = pd.read_csv(PACK / "trips.csv", parse_dates=["request_ts"])
customers = pd.read_csv(PACK / "customers.csv", parse_dates=["signup_date"])
drivers = pd.read_csv(PACK / "drivers.csv", parse_dates=["join_date"])
zones = pd.read_csv(PACK / "zones.csv")

print("Datasets loaded successfully ✓")

print("\nDataset Shapes:")
print("Trips:", trips.shape)
print("Customers:", customers.shape)
print("Drivers:", drivers.shape)
print("Zones:", zones.shape)

Libraries loaded ✓
Datasets loaded successfully ✓

Dataset Shapes:
Trips: (27226, 18)
Customers: (6000, 5)
Drivers: (1500, 6)
Zones: (24, 5)


## 2. Dataset Schema Inspection

In this section, we inspect the column names of all four datasets to understand their structure and identify the available features.

This step helps us determine which columns may contain information available at request time and which columns require further investigation for potential data leakage.

We will use the actual dataset columns to guide the leakage analysis rather than making assumptions based only on column names.

In [2]:
datasets = {
    "trips": trips,
    "customers": customers,
    "drivers": drivers,
    "zones": zones
}

for name, df in datasets.items():
    print(f"\n{'=' * 50}")
    print(f"{name.upper()} COLUMNS")
    print("=" * 50)

    for i, col in enumerate(df.columns, start=1):
        print(f"{i}. {col}")


TRIPS COLUMNS
1. trip_id
2. customer_id
3. driver_id
4. request_ts
5. pickup_zone
6. dropoff_zone
7. distance_km
8. surge_multiplier
9. est_fare
10. eta_estimate_min
11. weather
12. payment_method
13. service_level
14. request_failed
15. actual_trip_time_min
16. cancel_reason
17. support_ticket_flag
18. driver_arrival_min

CUSTOMERS COLUMNS
1. customer_id
2. signup_date
3. channel
4. home_zone
5. loyalty_tier

DRIVERS COLUMNS
1. driver_id
2. join_date
3. vehicle_type
4. experience_months
5. home_zone
6. rating_avg

ZONES COLUMNS
1. zone_id
2. zone_name
3. legacy_name
4. zone_type
5. has_metro


## 3. Trips Dataset Inspection

In this section, we examine the main `trips` dataset, which contains the ride request information and the target variable `request_failed`.

The inspection includes:

- **Dataset Information:** Check column data types and non-null counts.
- **First 10 Rows:** Preview the dataset to understand its structure and values.
- **Target Distribution:** Count successful (0) and failed (1) requests.
- **Missing Target Values:** Check whether the target column contains missing values.

This step provides an initial understanding of the prediction target and the data used in the leakage analysis.

In [3]:
print("Trips dataset information:")
trips.info()

print("\nFirst 10 rows:")
display(trips.head(10))

print("\nTarget column:")
print(trips["request_failed"].value_counts(dropna=False))

print("\nTarget missing values:")
print(trips["request_failed"].isna().sum())

Trips dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 27226 entries, 0 to 27225
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   trip_id               27226 non-null  int64         
 1   customer_id           27226 non-null  int64         
 2   driver_id             27226 non-null  int64         
 3   request_ts            27226 non-null  datetime64[us]
 4   pickup_zone           27226 non-null  str           
 5   dropoff_zone          27226 non-null  str           
 6   distance_km           27226 non-null  float64       
 7   surge_multiplier      27226 non-null  float64       
 8   est_fare              27226 non-null  float64       
 9   eta_estimate_min      27226 non-null  float64       
 10  weather               25913 non-null  str           
 11  payment_method        27226 non-null  str           
 12  service_level         27226 non-null  str           
 13  

,trip_id,customer_id,driver_id,request_ts,pickup_zone,dropoff_zone,distance_km,surge_multiplier,est_fare,eta_estimate_min,weather,payment_method,service_level,request_failed,actual_trip_time_min,cancel_reason,support_ticket_flag,driver_arrival_min
0,100001,21924,5062,2026-05-01 00:01:46,Dokki,Badr City,17.22,1.1,107.4,6.4,NaN,card,economy,0,29.9,NaN,0,5.1
1,100002,20391,5681,2026-05-01 00:37:36,Dokki,Obour,12.74,1.1,114.9,7.3,rain,cash,comfort,0,28.9,NaN,0,7.0
2,100003,24083,5407,2026-05-01 00:40:12,Maadi,Obour,10.80,1.0,92.3,6.4,NaN,card,comfort,0,22.5,NaN,0,4.0
3,100004,25342,5833,2026-05-01 00:49:23,Dokki,Mohandessin,9.51,1.2,75.5,8.1,rain,cash,economy,0,28.9,NaN,0,6.1
4,100005,25697,6119,2026-05-01 01:05:33,Dokki,Ramses,4.54,1.0,39.3,8.6,rain,cash,economy,0,20.6,NaN,0,9.7
5,100006,24471,6296,2026-05-01 01:26:18,Ramses,Ramses,4.91,1.0,44.5,8.8,NaN,cash,economy,0,28.7,NaN,0,7.6
6,100007,20066,5267,2026-05-01 01:26:54,Abdeen,Abdeen,5.06,1.0,45.2,9.4,rain,cash,economy,0,25.7,NaN,0,10.7
7,100008,21953,6016,2026-05-01 01:34:15,Tora,Maadi,17.52,1.1,104.1,7.9,NaN,wallet,economy,1,NaN,driver_rejected,0,NaN
8,100009,21953,6043,2026-05-01 01:36:28,Tora,Maadi,17.52,1.0,94.6,4.9,NaN,card,economy,1,NaN,no_driver_found,0,NaN
9,100010,25023,5874,2026-05-01 01:52:33,Nasr City,Dokki,5.80,1.0,45.1,8.9,rain,wallet,economy,0,22.6,NaN,0,7.5



Target column:
request_failed
0    21560
1     5666
Name: count, dtype: int64

Target missing values:
0


## 4. Missing Values Analysis by Target Class

In this section, we compare the percentage of missing values in each column between completed (0) and failed (1) ride requests.

The goal is to identify features whose missingness patterns are strongly associated with the target variable.

A significant difference in missing-value percentages may indicate potential data leakage, especially when a feature is only recorded after a ride is completed or fails.

The `Difference` column represents:

**Failed (1) Missing % − Completed (0) Missing %**

Positive values indicate more missing values in failed requests, while negative values indicate more missing values in completed requests.

This analysis helps identify suspicious features that require further investigation before model training.#%%


In [4]:
# Compare missing values for successful and failed requests

missing_comparison = trips.groupby("request_failed").apply(
    lambda group: group.isna().mean() * 100,
    include_groups=False
).T

missing_comparison.columns = ["Completed (0)", "Failed (1)"]

missing_comparison["Difference"] = (
    missing_comparison["Failed (1)"] -
    missing_comparison["Completed (0)"]
)

display(missing_comparison.round(2))

,Completed (0),Failed (1),Difference
trip_id,0.00,0.00,0.00
customer_id,0.00,0.00,0.00
driver_id,0.00,0.00,0.00
request_ts,0.00,0.00,0.00
pickup_zone,0.00,0.00,0.00
dropoff_zone,0.00,0.00,0.00
distance_km,0.00,0.00,0.00
surge_multiplier,0.00,0.00,0.00
est_fare,0.00,0.00,0.00
eta_estimate_min,0.00,0.00,0.00


## 5. Identification of Confirmed Data Leakage Features

Based on the missing values analysis, three columns were identified as confirmed data leakage features:

- **`actual_trip_time_min`:** Contains the actual trip duration, which is only available after the ride is completed.
- **`cancel_reason`:** Contains cancellation or failure information that directly reveals the request outcome.
- **`driver_arrival_min`:** Contains the actual driver arrival time, which is not available at the moment the request is created.

These features contain information that would not be available at prediction time. Therefore, they must be excluded from model training to prevent data leakage and ensure realistic evaluation.

The identified columns are stored in the `confirmed_leakage` list for documentation and subsequent preprocessing.

In [5]:
# Confirmed leakage columns

confirmed_leakage = [
    "actual_trip_time_min",
    "cancel_reason",
    "driver_arrival_min"
]

print("Confirmed Leakage Columns:")
for col in confirmed_leakage:
    print(f"- {col}")

print("\nThese columns must be excluded from model training.")

Confirmed Leakage Columns:
- actual_trip_time_min
- cancel_reason
- driver_arrival_min

These columns must be excluded from model training.


## 6. Investigation of Suspicious Features

In this section, we investigate two suspicious columns: `driver_id` and `support_ticket_flag`.

Although these columns do not contain missing values, their availability at prediction time must be verified.

The analysis includes:

- **Value Counts by Target:** Examines how each feature's values are distributed across completed and failed requests.
- **Failure Rate by Value:** Calculates the proportion of failed requests for each value to identify unusual patterns or strong associations with the target.

**Columns under investigation:**

- **`driver_id`:** Determines whether a driver is assigned at request creation or later in the ride process. If assignment happens after the request, this feature must be excluded.
- **`support_ticket_flag`:** Checks whether support tickets are associated with failed requests. If tickets are created after a failure occurs, this feature contains future information and must be excluded.

The results will help determine whether these features can safely be used for model training.

In [6]:
# Investigate suspicious columns

suspicious_columns = [
    "driver_id",
    "support_ticket_flag"
]

for col in suspicious_columns:

    print(f"\n{'=' * 50}")
    print(f"Column: {col}")
    print("=" * 50)

    print("\nValue counts by target:")
    display(
        pd.crosstab(
            trips[col],
            trips["request_failed"],
            margins=True
        )
    )

    print("\nFailure rate by value:")
    display(
        trips.groupby(col)["request_failed"]
        .agg(["count", "mean"])
        .rename(columns={"mean": "failure_rate"})
        .round(3)
    )


Column: driver_id

Value counts by target:


request_failed,0,1,All
driver_id,,,
5000,17,5,22
5001,28,7,35
5002,9,2,11
5003,26,6,32
5004,22,6,28
...,...,...,...
6496,14,4,18
6497,39,4,43
6498,16,1,17



Failure rate by value:


,count,failure_rate
driver_id,,
5000,22,0.227
5001,35,0.200
5002,11,0.182
5003,32,0.188
5004,28,0.214
...,...,...
6495,17,0.294
6496,18,0.222
6497,43,0.093



Column: support_ticket_flag

Value counts by target:


request_failed,0,1,All
support_ticket_flag,,,
0,21132,4275,25407
1,428,1391,1819
All,21560,5666,27226



Failure rate by value:


,count,failure_rate
support_ticket_flag,,
0,25407,0.168
1,1819,0.765


### 6.1 Suspicious Features Analysis Results

**1. driver_id**

The analysis shows that failure rates vary across drivers. For example, driver 6499 has a failure rate of 40%, while driver 6498 has a failure rate of 5.9%.

However, this variation does not prove data leakage. The main concern is whether the driver is assigned before or after the prediction time.

**Decision: INVESTIGATE.** The feature should only be retained if driver assignment is confirmed to be available at request creation.

**2. support_ticket_flag**

Requests with `support_ticket_flag = 1` have a failure rate of 76.5%, compared with 16.8% for requests with `support_ticket_flag = 0`.

This strong association raises suspicion that the feature may contain information recorded after a request fails or encounters a problem.

**Decision: POTENTIAL TEMPORAL LEAKAGE.** Exclude this feature if support tickets are created after the request outcome becomes known. Its recording time must be verified before making the final decision.

In [7]:
# Merge driver information with trips

driver_check = trips[
    ["trip_id", "driver_id", "request_ts", "request_failed"]
].merge(
    drivers[["driver_id", "join_date"]],
    on="driver_id",
    how="left"
)

# Check drivers who joined after the request date

driver_check["joined_after_request"] = (
    driver_check["join_date"] > driver_check["request_ts"]
)

print("Driver join-date consistency check:")

print(
    driver_check["joined_after_request"]
    .value_counts(dropna=False)
)

print("\nJoined after request by target:")
display(
    pd.crosstab(
        driver_check["joined_after_request"],
        driver_check["request_failed"],
        margins=True
    )
)

Driver join-date consistency check:
joined_after_request
False    27226
Name: count, dtype: int64

Joined after request by target:


request_failed,0,1,All
joined_after_request,,,
False,21560,5666,27226
All,21560,5666,27226


### 6.2 Driver Join-Date Consistency Check

The driver join-date consistency check was performed by comparing each driver's `join_date` with the corresponding trip's `request_ts`.

**Results:**
- Total requests checked: 27,226.
- Drivers who joined after the request date: 0.
- Requests with valid driver join-date matches: 27,226.

All assigned drivers had joined the platform before their corresponding requests were created.

**Conclusion:** No temporal inconsistency was detected in driver join dates. However, this does not confirm that `driver_id` was available at request creation, because the actual driver assignment timestamp is not provided.

**Decision: INVESTIGATE.** Retain `driver_id` only if the assignment process confirms that the driver is known at prediction time.

In [25]:
# Check customer and driver feature availability

print("=" * 50)
print("CUSTOMER FEATURES")
print("=" * 50)

customer_features = [
    "signup_date",
    "channel",
    "home_zone",
    "loyalty_tier"
]

display(customers[customer_features].describe(include="all").T)

print("\nMissing values:")
display(customers[customer_features].isna().sum())


print("\n" + "=" * 50)
print("DRIVER FEATURES")
print("=" * 50)

driver_features = [
    "join_date",
    "vehicle_type",
    "experience_months",
    "home_zone",
    "rating_avg"
]

display(drivers[driver_features].describe(include="all").T)

print("\nMissing values:")
display(drivers[driver_features].isna().sum())

CUSTOMER FEATURES


,count,unique,top,freq,mean,min,25%,50%,75%,max
signup_date,6000,NaN,NaN,NaN,2026-03-05 05:33:36,2025-10-14 00:00:00,2026-01-08 00:00:00,2026-03-05 00:00:00,2026-04-25 00:00:00,2026-09-24 00:00:00
channel,6000,3,organic,2986,NaN,NaN,NaN,NaN,NaN,NaN
home_zone,6000,24,Nasr City,437,NaN,NaN,NaN,NaN,NaN,NaN
loyalty_tier,6000,3,bronze,3000,NaN,NaN,NaN,NaN,NaN,NaN



Missing values:


signup_date     0
channel         0
home_zone       0
loyalty_tier    0
dtype: int64


DRIVER FEATURES


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
join_date,1500,NaN,NaN,NaN,2026-03-16 14:21:07.200000,2025-12-17 00:00:00,2026-02-13 00:00:00,2026-03-11 00:00:00,2026-04-06 00:00:00,2026-08-17 00:00:00,NaN
vehicle_type,1500,3,sedan,871,NaN,NaN,NaN,NaN,NaN,NaN,NaN
experience_months,1500.0,NaN,NaN,NaN,38.52,1.0,19.0,32.0,52.0,140.0,26.365502
home_zone,1500,24,Nasr City,110,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rating_avg,1238.0,NaN,NaN,NaN,4.546502,3.92,4.42,4.54,4.66,5.0,0.184513



Missing values:


join_date              0
vehicle_type           0
experience_months      0
home_zone              0
rating_avg           262
dtype: int64

In [26]:
# Merge customer information with trips

customer_check = trips[
    ["trip_id", "customer_id", "request_ts", "request_failed"]
].merge(
    customers[["customer_id", "signup_date"]],
    on="customer_id",
    how="left"
)

# Check customers who signed up after the request

customer_check["signup_after_request"] = (
    customer_check["signup_date"] > customer_check["request_ts"]
)

print("Customer signup-date consistency:")

print(
    customer_check["signup_after_request"]
    .value_counts(dropna=False)
)

print("\nSignup after request by target:")

display(
    pd.crosstab(
        customer_check["signup_after_request"],
        customer_check["request_failed"],
        margins=True
    )
)

Customer signup-date consistency:
signup_after_request
False    27226
Name: count, dtype: int64

Signup after request by target:


request_failed,0,1,All
signup_after_request,,,
False,21560,5666,27226
All,21560,5666,27226


### 6.3 Customer Signup-Date Consistency Check

The customer signup-date consistency check was performed by comparing each customer's `signup_date` with the corresponding trip's `request_ts`.

**Results:**
- Total requests checked: 27,226.
- Customers who signed up after the request: 0.
- Requests with valid customer signup dates: 27,226.

All customers had registered on the platform before creating their corresponding ride requests.

**Conclusion:** No temporal inconsistency was detected in customer signup dates.

**Decision:** `signup_date` is temporally consistent with the prediction time. However, other customer attributes, such as `loyalty_tier`, should still be checked to ensure that their values were available at request creation.

In [28]:
# Customer and driver feature statistics

print("CUSTOMER FEATURES")
display(customers.describe(include="all").T)

print("\nDRIVER FEATURES")
display(drivers.describe(include="all").T)

print("\nCustomer categorical values:")
for col in ["channel", "home_zone", "loyalty_tier"]:
    print(f"\n{col}:")
    print(customers[col].value_counts(dropna=False))

print("\nDriver categorical values:")
for col in ["vehicle_type", "home_zone"]:
    print(f"\n{col}:")
    print(drivers[col].value_counts(dropna=False))

CUSTOMER FEATURES


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
customer_id,6000.0,NaN,NaN,NaN,22999.5,20000.0,21499.75,22999.5,24499.25,25999.0,1732.195139
signup_date,6000,NaN,NaN,NaN,2026-03-05 05:33:36,2025-10-14 00:00:00,2026-01-08 00:00:00,2026-03-05 00:00:00,2026-04-25 00:00:00,2026-09-24 00:00:00,NaN
channel,6000,3,organic,2986,NaN,NaN,NaN,NaN,NaN,NaN,NaN
home_zone,6000,24,Nasr City,437,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loyalty_tier,6000,3,bronze,3000,NaN,NaN,NaN,NaN,NaN,NaN,NaN



DRIVER FEATURES


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
driver_id,1500.0,NaN,NaN,NaN,5749.5,5000.0,5374.75,5749.5,6124.25,6499.0,433.157015
join_date,1500,NaN,NaN,NaN,2026-03-16 14:21:07.200000,2025-12-17 00:00:00,2026-02-13 00:00:00,2026-03-11 00:00:00,2026-04-06 00:00:00,2026-08-17 00:00:00,NaN
vehicle_type,1500,3,sedan,871,NaN,NaN,NaN,NaN,NaN,NaN,NaN
experience_months,1500.0,NaN,NaN,NaN,38.52,1.0,19.0,32.0,52.0,140.0,26.365502
home_zone,1500,24,Nasr City,110,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rating_avg,1238.0,NaN,NaN,NaN,4.546502,3.92,4.42,4.54,4.66,5.0,0.184513



Customer categorical values:

channel:
channel
organic     2986
ads         2119
referral     895
Name: count, dtype: int64

home_zone:
home_zone
Nasr City        437
Dokki            401
Shubra           399
Heliopolis       364
Mohandessin      350
Maadi            337
Abdeen           306
Agouza           298
Garden City      276
Shorouk          273
Ramses           256
Zamalek          247
Downtown         240
Cairo Airport    235
Obour            208
New Cairo        204
Madinaty         193
6th October      191
Qalyub           183
Sheikh Zayed     144
Helwan           133
Khanka           112
Badr City        108
Tora             105
Name: count, dtype: int64

loyalty_tier:
loyalty_tier
bronze    3000
silver    1800
gold      1200
Name: count, dtype: int64

Driver categorical values:

vehicle_type:
vehicle_type
sedan        871
hatchback    458
van          171
Name: count, dtype: int64

home_zone:
home_zone
Nasr City        110
Heliopolis       107
Shubra           103
Mohand

### 6.4 Customer and Driver Feature Inspection

The customer and driver datasets were examined to identify potentially useful features and detect possible temporal leakage.

**Customer features:**

- `signup_date`: All customers registered before their corresponding requests.
- `channel` and `home_zone`: Categorical attributes that may be available at request time.
- `loyalty_tier`: Requires further investigation because customer loyalty levels may change over time.
- `customer_id`: An identifier rather than a meaningful behavioral feature.

**Driver features:**

- `join_date`: All drivers joined the platform before their corresponding requests.
- `vehicle_type` and `home_zone`: Potentially usable driver attributes.
- `experience_months`: Requires verification of the date used to calculate driver experience.
- `rating_avg`: Requires verification that ratings were calculated using only information available before the request.
- `driver_id`: Requires confirmation that the driver was assigned at prediction time.

**Conclusion:** Customer and driver attributes may provide useful predictive information, but their historical availability must be verified before including them in model training.

In [11]:
# Check missing values in customer and driver features

print("CUSTOMER FEATURE MISSING VALUES")
display(customers.isna().sum())

print("\nDRIVER FEATURE MISSING VALUES")
display(drivers.isna().sum())


# Compare driver experience with platform tenure at request time

driver_temporal = trips[
    ["trip_id", "driver_id", "request_ts"]
].merge(
    drivers[["driver_id", "join_date", "experience_months"]],
    on="driver_id",
    how="left"
)

driver_temporal["platform_tenure_months"] = (
    (driver_temporal["request_ts"] - driver_temporal["join_date"])
    .dt.days / 30.44
)

print("\nDriver experience compared with platform tenure:")
display(
    driver_temporal[
        ["experience_months", "platform_tenure_months"]
    ].describe().round(2)
)

print("\nDrivers with experience lower than platform tenure:")
print(
    (
        driver_temporal["experience_months"]
        < driver_temporal["platform_tenure_months"]
    ).sum()
)

CUSTOMER FEATURE MISSING VALUES


customer_id     0
signup_date     0
channel         0
home_zone       0
loyalty_tier    0
dtype: int64


DRIVER FEATURE MISSING VALUES


driver_id              0
join_date              0
vehicle_type           0
experience_months      0
home_zone              0
rating_avg           262
dtype: int64


Driver experience compared with platform tenure:


,experience_months,platform_tenure_months
count,27226.00,27226.00
mean,37.97,3.69
std,25.97,1.56
min,1.00,0.00
25%,19.00,2.56
50%,32.00,3.71
75%,52.00,4.83
max,140.00,8.28



Drivers with experience lower than platform tenure:
196


### 6.5 Customer and Driver Missing Values Analysis

The missing-value analysis showed that all customer features have complete data.

In the driver dataset, `rating_avg` contains 262 missing values out of 1,500 drivers, representing approximately 17.47%.

This missingness does not directly indicate data leakage, but it may reflect drivers who have not received enough ratings.

### 6.6 Driver Experience Temporal Consistency

Driver experience was compared with platform tenure at the time of each request.

**Results:**
- Average driving experience: 37.97 months.
- Average platform tenure: 3.69 months.
- Minimum driving experience: 1 month.
- Maximum driving experience: 140 months.
- Average platform tenure is considerably shorter than average driving experience.

This difference is reasonable because drivers may have professional driving experience before joining the platform.

**Conclusion:** No obvious temporal inconsistency was identified from this comparison.

**Decision:** `experience_months` can be considered for model training, provided it represents experience available at request time. The calculation reference date should still be verified.

In [12]:
# Merge customer and driver attributes with trips

trips_with_profiles = trips.merge(
    customers[["customer_id", "loyalty_tier"]],
    on="customer_id",
    how="left"
).merge(
    drivers[["driver_id", "rating_avg"]],
    on="driver_id",
    how="left"
)

# Check loyalty tier distribution by target

print("Loyalty tier by request outcome:")
display(
    pd.crosstab(
        trips_with_profiles["loyalty_tier"],
        trips_with_profiles["request_failed"],
        margins=True
    )
)

print("\nFailure rate by loyalty tier:")
display(
    trips_with_profiles.groupby("loyalty_tier")["request_failed"]
    .agg(["count", "mean"])
    .rename(columns={"mean": "failure_rate"})
    .round(3)
)

# Check missing driver ratings by target

print("\nRating missingness by target:")
display(
    pd.crosstab(
        trips_with_profiles["rating_avg"].isna(),
        trips_with_profiles["request_failed"],
        margins=True
    )
)

# Compare rating statistics by target

print("\nRating statistics by target:")
display(
    trips_with_profiles.groupby("request_failed")["rating_avg"]
    .agg(["count", "mean", "min", "max"])
    .round(3)
)

Loyalty tier by request outcome:


request_failed,0,1,All
loyalty_tier,,,
bronze,3455,1625,5080
gold,11014,2157,13171
silver,7091,1884,8975
All,21560,5666,27226



Failure rate by loyalty tier:


,count,failure_rate
loyalty_tier,,
bronze,5080,0.320
gold,13171,0.164
silver,8975,0.210



Rating missingness by target:


request_failed,0,1,All
rating_avg,,,
False,19478,4946,24424
True,2082,720,2802
All,21560,5666,27226



Rating statistics by target:


,count,mean,min,max
request_failed,,,,
0,19478,4.559,3.92,5.0
1,4946,4.556,3.92,5.0


### 6.7 Loyalty Tier and Driver Rating Analysis

**Loyalty Tier Analysis**

The failure rates differ across customer loyalty tiers:
- Bronze: 32.0%
- Gold: 16.4%
- Silver: 21.0%

This indicates an association between loyalty tier and request failure. However, this association does not necessarily indicate data leakage.

**Decision:** `loyalty_tier` can be considered for model training, provided that the recorded tier represents the customer's status at request creation. Its update and snapshot timing should be verified.

**Driver Rating Analysis**

The average driver rating is nearly identical for completed and failed requests (4.559 and 4.556, respectively).

However, missing ratings show a higher failure rate:
- Missing rating: approximately 25.7%.
- Available rating: approximately 20.3%.

This difference may reflect differences between drivers with and without recorded ratings, but it does not independently confirm data leakage.

**Decision:** `rating_avg` can be considered for model training if it is calculated exclusively from information available before the request timestamp


In [13]:
# Inspect zone attributes

print("Zones dataset information:")
zones.info()

print("\nFirst 10 rows:")
display(zones.head(10))

print("\nZone type distribution:")
display(zones["zone_type"].value_counts(dropna=False))

print("\nMetro availability:")
display(zones["has_metro"].value_counts(dropna=False))

print("\nMissing values:")
display(zones.isnull().sum())

Zones dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   zone_id      24 non-null     str  
 1   zone_name    24 non-null     str  
 2   legacy_name  24 non-null     str  
 3   zone_type    24 non-null     str  
 4   has_metro    24 non-null     int64
dtypes: int64(1), str(4)
memory usage: 1.1 KB

First 10 rows:


,zone_id,zone_name,legacy_name,zone_type,has_metro
0,Z01,Downtown,Downtown,downtown,1
1,Z02,Zamalek,Zamalek,downtown,1
2,Z03,Garden City,Garden City,downtown,1
3,Z04,Abdeen,Abdeen,downtown,1
4,Z05,Shubra District,Shubra,downtown,1
5,Z06,Ramses,Ramses,downtown,1
6,Z07,Maadi,Maadi,midtown,1
7,Z08,Nasr City,Nasr City,midtown,1
8,Z09,Heliopolis,Heliopolis,midtown,1
9,Z10,Mohandessin,Mohandessin,midtown,0



Zone type distribution:


zone_type
downtown    6
midtown     6
new_city    6
suburb      5
airport     1
Name: count, dtype: int64


Metro availability:


has_metro
0    15
1     9
Name: count, dtype: int64


Missing values:


zone_id        0
zone_name      0
legacy_name    0
zone_type      0
has_metro      0
dtype: int64

### 6.8 Zones Dataset Inspection

The zones dataset contains 24 records and 5 attributes describing geographical areas.

No missing values were identified.

The attributes include:
- `zone_id`: Unique identifier used to connect zones with trip records.
- `zone_name` and `legacy_name`: Descriptive zone names.
- `zone_type`: Categorical classification of each zone.
- `has_metro`: Binary indicator of metro availability.

**Initial Leakage Assessment:**

These attributes represent geographical characteristics rather than information generated from the outcome of a ride. No

In [14]:
# Build a zone lookup that recognizes both current and legacy zone labels.
# This avoids losing valid matches when trips use names such as "Shubra"
# while the reference table stores "Shubra District" as zone_name.

zone_lookup = pd.concat([
    zones[["zone_name", "zone_type", "has_metro"]].rename(
        columns={"zone_name": "zone_label"}
    ),
    zones[["legacy_name", "zone_type", "has_metro"]].rename(
        columns={"legacy_name": "zone_label"}
    )
], ignore_index=True).drop_duplicates(subset=["zone_label"])

# Normalize labels only for matching; preserve the original trip columns.
zone_lookup["zone_key"] = (
    zone_lookup["zone_label"].astype(str).str.strip().str.casefold()
)

trips_zones = trips.copy()
trips_zones["pickup_zone_key"] = (
    trips_zones["pickup_zone"].astype(str).str.strip().str.casefold()
)
trips_zones["dropoff_zone_key"] = (
    trips_zones["dropoff_zone"].astype(str).str.strip().str.casefold()
)

pickup_lookup = zone_lookup[
    ["zone_key", "zone_type", "has_metro"]
].rename(columns={
    "zone_type": "pickup_zone_type",
    "has_metro": "pickup_has_metro"
})

dropoff_lookup = zone_lookup[
    ["zone_key", "zone_type", "has_metro"]
].rename(columns={
    "zone_type": "dropoff_zone_type",
    "has_metro": "dropoff_has_metro"
})

trips_zones = trips_zones.merge(
    pickup_lookup, left_on="pickup_zone_key", right_on="zone_key",
    how="left", validate="many_to_one"
).drop(columns=["pickup_zone_key", "zone_key"])

trips_zones["dropoff_zone_key"] = (
    trips_zones["dropoff_zone"].astype(str).str.strip().str.casefold()
)
trips_zones = trips_zones.merge(
    dropoff_lookup, left_on="dropoff_zone_key", right_on="zone_key",
    how="left", validate="many_to_one"
).drop(columns=["dropoff_zone_key", "zone_key"])

print("Missing pickup zone attributes:",
      trips_zones["pickup_zone_type"].isna().sum())
print("Missing drop-off zone attributes:",
      trips_zones["dropoff_zone_type"].isna().sum())
print("Original rows:", len(trips), "| Rows after merges:", len(trips_zones))


Missing pickup zone attributes: 0
Missing drop-off zone attributes: 0
Original rows: 27226 | Rows after merges: 27226


In [15]:
# Validate the zone enrichment and inspect any labels that remain unmatched.
print("Trips shape:", trips.shape)
print("Merged data shape:", trips_zones.shape)

print("\nPickup zone types:")
display(trips_zones["pickup_zone_type"].value_counts(dropna=False))

print("\nDrop-off zone types:")
display(trips_zones["dropoff_zone_type"].value_counts(dropna=False))

unmatched_pickup = sorted(
    trips.loc[trips_zones["pickup_zone_type"].isna(), "pickup_zone"]
    .dropna().unique()
)
unmatched_dropoff = sorted(
    trips.loc[trips_zones["dropoff_zone_type"].isna(), "dropoff_zone"]
    .dropna().unique()
)

print("\nUnmatched pickup labels:", unmatched_pickup)
print("Unmatched drop-off labels:", unmatched_dropoff)

assert len(trips_zones) == len(trips), (
    "Zone merge changed the number of trip rows; investigate duplicate keys."
)


Trips shape: (27226, 18)
Merged data shape: (27226, 22)

Pickup zone types:


pickup_zone_type
midtown     9611
downtown    7382
new_city    5955
suburb      3137
airport     1141
Name: count, dtype: int64


Drop-off zone types:


dropoff_zone_type
downtown    9941
midtown     9647
new_city    4332
suburb      1751
airport     1555
Name: count, dtype: int64


Unmatched pickup labels: []
Unmatched drop-off labels: []


## 7. Additional Data Leakage Checks

The previous analysis identified clear and potentially suspicious columns, but those checks alone are not sufficient to establish that the dataset is leakage-free. This section examines:

1. Duplicate records and identifier integrity.
2. Relationships between trips, customers, and drivers.
3. Target distribution over time, which may reveal temporal patterns or data shifts.
4. Features that may reveal the outcome directly or indirectly.
5. Whether each feature would be available at the time a request is created.

**Important:** A strong association with `request_failed` does not, by itself, prove data leakage. The key question is whether the information was available at prediction time.


In [24]:


print("\nTarget values:")
display(trips["request_failed"].value_counts(dropna=False).sort_index())

print("\nTarget proportions:")
display((trips["request_failed"].value_counts(normalize=True) * 100).round(2))

# A trip identifier should identify one row.
assert trips["trip_id"].is_unique, "trip_id is not unique."
assert trips["request_failed"].notna().all(), "Target contains missing values."
assert set(trips["request_failed"].dropna().unique()).issubset({0, 1}), (
    "Unexpected target labels found."
)


Target values:


request_failed
0    21560
1     5666
Name: count, dtype: int64


Target proportions:


request_failed
0    79.19
1    20.81
Name: proportion, dtype: float64

### Interpreting the Duplicate Checks

- `trip_id` should uniquely identify each trip. Duplicates may overrepresent certain cases or allow copies of the same trip to appear in both training and test sets.
- `customer_id` should be unique in the customers table, and `driver_id` should be unique in the drivers table.
- A unique identifier is not automatically leakage, but it usually does not represent a generalizable behavioral feature. Do not use it as a predictor without a clear justification.


In [17]:
# 7.2 Referential integrity: do all trip IDs exist in the profile tables?

missing_customer_profiles = (~trips["customer_id"].isin(customers["customer_id"])).sum()
missing_driver_profiles = (~trips["driver_id"].isin(drivers["driver_id"])).sum()

print("Trips without a matching customer profile:", missing_customer_profiles)
print("Trips without a matching driver profile:", missing_driver_profiles)

# Verify many-to-one relationships before profile merges.
print("\nCustomer key unique:", customers["customer_id"].is_unique)
print("Driver key unique:", drivers["driver_id"].is_unique)

Trips without a matching customer profile: 0
Trips without a matching driver profile: 0

Customer key unique: True
Driver key unique: True


## 8. Temporal Analysis and Potential Time Leakage

We inspect the date range and failure rates by day and hour. These checks do not prove leakage, but they can reveal temporal shifts or non-random ordering that may affect the train/test split.

When building the model, the split strategy should match the intended real-world prediction scenario. If the goal is to predict future requests, a chronological split is generally more appropriate than randomly mixing all time periods.


In [18]:
# 8.1 Request timestamp range and chronological ordering

print("First request:", trips["request_ts"].min())
print("Last request:", trips["request_ts"].max())
print("Rows already sorted by request_ts:",
      trips["request_ts"].is_monotonic_increasing)

time_audit = trips[["request_ts", "request_failed"]].copy()
time_audit["request_date"] = time_audit["request_ts"].dt.date
time_audit["request_hour"] = time_audit["request_ts"].dt.hour
time_audit["request_dayofweek"] = time_audit["request_ts"].dt.day_name()

daily_failure = (
    time_audit.groupby("request_date")["request_failed"]
    .agg(requests="count", failure_rate="mean")
)
daily_failure["failure_rate_pct"] = daily_failure["failure_rate"] * 100

hourly_failure = (
    time_audit.groupby("request_hour")["request_failed"]
    .agg(requests="count", failure_rate="mean")
)
hourly_failure["failure_rate_pct"] = hourly_failure["failure_rate"] * 100

print("\nDaily failure-rate summary:")
display(daily_failure.describe().round(3))

print("\nHourly failure rates:")
display(hourly_failure.round(3))

First request: 2026-05-01 00:01:46
Last request: 2026-08-27 23:57:39
Rows already sorted by request_ts: True

Daily failure-rate summary:


,requests,failure_rate,failure_rate_pct
count,119.000,119.000,119.000
mean,228.790,0.208,20.765
std,25.419,0.033,3.349
min,170.000,0.126,12.560
25%,210.000,0.184,18.389
50%,230.000,0.209,20.939
75%,245.000,0.229,22.857
max,295.000,0.295,29.524



Hourly failure rates:


,requests,failure_rate,failure_rate_pct
request_hour,,,
0,867,0.306,30.565
1,765,0.282,28.235
2,593,0.261,26.138
3,471,0.278,27.813
4,338,0.249,24.852
5,369,0.268,26.829
6,765,0.299,29.935
7,1699,0.201,20.071
8,2197,0.077,7.692


### Interpreting the Temporal Results

Differences in failure rates across hours or days may reflect genuine demand patterns rather than leakage. However, if row order or `trip_id` reflects time, the split should not accidentally allow the model to learn from future records and evaluate on past records.

We do not classify `trip_id` or `request_ts` as leakage merely because they correlate with ordering. Their use depends on the prediction scenario. Derived time features should only be used if they are known at the moment the prediction is made.


In [19]:
# 8.2 Check whether trip_id follows request time.
# This is a diagnostic only; it does not establish that the ID should be a feature.

id_time = trips[["trip_id", "request_ts"]].sort_values("trip_id")
print("Trip ID increases with request time:",
      id_time["request_ts"].is_monotonic_increasing)

print("\nFirst and last records after sorting by trip_id:")
display(pd.concat([id_time.head(3), id_time.tail(3)]))

Trip ID increases with request time: True

First and last records after sorting by trip_id:


,trip_id,request_ts
0,100001,2026-05-01 00:01:46
1,100002,2026-05-01 00:37:36
2,100003,2026-05-01 00:40:12
27223,127224,2026-08-27 23:48:19
27224,127225,2026-08-27 23:49:51
27225,127226,2026-08-27 23:57:39


## 9. Feature Audit Based on Prediction Time

For this audit, we assume the prediction is made **when the trip request is created**, at `request_ts`.

Under this assumption:

- Any information generated after the trip is completed or fails must be excluded.
- Information that may exist before the request but has no historical creation timestamp is classified as requiring verification.
- Association with the target must not be confused with confirmed leakage.


In [20]:
# 9.1 Consolidated feature-level leakage decisions

feature_decisions = pd.DataFrame([
    ("trip_id", "Identifier", "Exclude initially", "Technical identifier; may encode temporal ordering and is not a generalizable behavioral feature."),
    ("customer_id", "Identifier", "Exclude initially", "Customer identifier; do not use as a raw feature without justification."),
    ("driver_id", "Requires verification", "Conditional", "Verify that the driver had been assigned before prediction time; its presence in the row does not prove this."),
    ("request_ts", "Request timestamp", "Use derived features only", "Known at request time; hour or day features may be derived if appropriate."),
    ("pickup_zone", "Location", "Provisionally allowed", "Pickup location is generally available when the request is created."),
    ("dropoff_zone", "Location", "Provisionally allowed", "Destination is generally available when the request is created."),
    ("distance_km", "Route estimate", "Verify", "Confirm that this is an estimated distance available at request time, not an actual distance measured after the trip."),
    ("surge_multiplier", "Pricing", "Allowed if verified", "Must be the pricing value calculated at request time."),
    ("est_fare", "Fare estimate", "Allowed if verified", "Must be the estimated fare at request time, not the final fare."),
    ("eta_estimate_min", "Estimate", "Allowed if verified", "The ETA must be available at request time."),
    ("weather", "Context", "Allowed if verified", "Must represent weather at request time; also review missing values."),
    ("payment_method", "User selection", "Provisionally allowed", "Allowed if selected by the user before submitting the request."),
    ("service_level", "User selection", "Provisionally allowed", "Allowed if the service level is selected before prediction."),
    ("request_failed", "Target", "Never include in X", "This is the variable being predicted."),
    ("actual_trip_time_min", "Post-event", "Confirmed leakage", "Actual duration is not known when the request is created."),
    ("cancel_reason", "Outcome / failure reason", "Confirmed leakage", "Reveals the cancellation or failure reason after the event."),
    ("support_ticket_flag", "Likely post-event", "Exclude until verified", "Strongly associated with failure and may be created after the issue; the ticket timestamp must be checked."),
    ("driver_arrival_min", "Post-request", "Confirmed leakage", "Actual arrival time is not known when the request is created."),
    ("signup_date", "Customer profile", "Allowed if verified", "The signup date predates the request in the current check; use account age as of the request time."),
    ("channel", "Customer profile", "Allowed if verified", "Confirm that the customer acquisition channel is stable and known at request time."),
    ("home_zone", "Customer / driver profile", "Allowed if verified", "Verify that the recorded location is the version available at request time."),
    ("loyalty_tier", "Customer profile", "Requires verification", "May change over time; a historical snapshot as of request_ts is required."),
    ("join_date", "Driver profile", "Allowed if verified", "The join date predates the request in the current check; tenure can be derived as of the request time."),
    ("vehicle_type", "Driver profile", "Requires verification", "Verify that the vehicle type was known for the assigned driver at request time."),
    ("experience_months", "Driver profile", "Requires verification", "The reference date used to calculate experience must be known."),
    ("rating_avg", "Driver profile", "Requires verification", "The average must be calculated using ratings recorded before the request only."),
    ("zone_type / has_metro", "Geographic reference", "Provisionally allowed", "Relatively stable geographic attributes; verify zone-name matching.")
], columns=["feature", "category", "decision", "reason"])

display(feature_decisions)

,feature,category,decision,reason
0,trip_id,Identifier,Exclude initially,Technical identifier; may encode temporal orde...
1,customer_id,Identifier,Exclude initially,Customer identifier; do not use as a raw featu...
2,driver_id,Requires verification,Conditional,Verify that the driver had been assigned befor...
3,request_ts,Request timestamp,Use derived features only,Known at request time; hour or day features ma...
4,pickup_zone,Location,Provisionally allowed,Pickup location is generally available when th...
5,dropoff_zone,Location,Provisionally allowed,Destination is generally available when the re...
6,distance_km,Route estimate,Verify,Confirm that this is an estimated distance ava...
7,surge_multiplier,Pricing,Allowed if verified,Must be the pricing value calculated at reques...
8,est_fare,Fare estimate,Allowed if verified,"Must be the estimated fare at request time, no..."
9,eta_estimate_min,Estimate,Allowed if verified,The ETA must be available at request time.


## 10. Investigating Suspicious Feature Associations

We compare failure rates across categories for selected sensitive features. The purpose is to document signals that warrant checking when the data was created, not to automatically remove every feature associated with the target.


In [21]:
# 10.1 Target association for selected categorical / binary features

association_columns = [
    col for col in ["support_ticket_flag", "payment_method",
                    "service_level", "weather", "loyalty_tier"]
    if col in trips.columns or col in customers.columns
]

# Add loyalty tier only for this diagnostic merge.
association_data = trips.merge(
    customers[["customer_id", "loyalty_tier"]],
    on="customer_id", how="left", validate="many_to_one"
)

for col in ["support_ticket_flag", "payment_method", "service_level",
            "weather", "loyalty_tier"]:
    if col not in association_data.columns:
        continue
    print(f"\nFailure rate by {col}:")
    display(
        association_data.groupby(col, dropna=False)["request_failed"]
        .agg(requests="size", failure_rate="mean")
        .assign(failure_rate_pct=lambda x: x["failure_rate"] * 100)
        .round(3)
        .sort_values("failure_rate", ascending=False)
    )


Failure rate by support_ticket_flag:


,requests,failure_rate,failure_rate_pct
support_ticket_flag,,,
1,1819,0.765,76.471
0,25407,0.168,16.826



Failure rate by payment_method:


,requests,failure_rate,failure_rate_pct
payment_method,,,
cash,14228,0.221,22.083
card,10126,0.196,19.603
wallet,2872,0.188,18.767



Failure rate by service_level:


,requests,failure_rate,failure_rate_pct
service_level,,,
economy,19150,0.260,25.990
comfort,5926,0.101,10.074
premium,2150,0.043,4.279



Failure rate by weather:


,requests,failure_rate,failure_rate_pct
weather,,,
rain,1661,0.240,24.022
NaN,1313,0.239,23.915
dust,1858,0.223,22.282
clear,22394,0.203,20.269



Failure rate by loyalty_tier:


,requests,failure_rate,failure_rate_pct
loyalty_tier,,,
bronze,5080,0.320,31.988
silver,8975,0.210,20.992
gold,13171,0.164,16.377


## 11. Leakage Prevention Rules for Model Development

Apply the following rules during the Modeling phase as well:

1. Separate `request_failed` from the feature columns; never include the target in `X`.
2. Remove confirmed leakage columns. Exclude `support_ticket_flag` and `driver_id` until their availability timestamps have been verified.
3. Do not use `actual_trip_time_min` or `driver_arrival_min` to impute missing values; their absence may be related to the trip outcome.
4. Perform imputation, categorical encoding, and scaling inside a Pipeline fitted on training data only.
5. If evaluation concerns future requests, use a chronological split. Do not use test data for feature selection or hyperparameter tuning.
6. When joining customer and driver profiles, verify that they are historical snapshots as of the request time, not the latest versions collected at the end of the observation period.


In [22]:
# 11.1 Recommended initial exclusion list for model preparation.
# Keep this list conservative until the unresolved timestamps are confirmed.

confirmed_leakage = [
    "actual_trip_time_min",
    "cancel_reason",
    "driver_arrival_min"
]

exclude_until_verified = [
    "support_ticket_flag",
    "driver_id"
]

identifier_columns = [
    "trip_id",
    "customer_id"
]

target_column = "request_failed"

print("Confirmed leakage:", confirmed_leakage)
print("Exclude until timing is verified:", exclude_until_verified)
print("Identifiers not recommended as raw predictors:", identifier_columns)
print("Target:", target_column)

# Construct a safe trip-level starting frame for the next modeling phase.
# This is not the final feature matrix: additional profile columns must be
# merged only after historical availability is verified.
initial_excluded = set(
    confirmed_leakage + exclude_until_verified +
    identifier_columns + [target_column]
)
candidate_trip_features = [
    c for c in trips.columns if c not in initial_excluded
]

print("\nCandidate trip-level columns (subject to availability checks):")
print(candidate_trip_features)

Confirmed leakage: ['actual_trip_time_min', 'cancel_reason', 'driver_arrival_min']
Exclude until timing is verified: ['support_ticket_flag', 'driver_id']
Identifiers not recommended as raw predictors: ['trip_id', 'customer_id']
Target: request_failed

Candidate trip-level columns (subject to availability checks):
['request_ts', 'pickup_zone', 'dropoff_zone', 'distance_km', 'surge_multiplier', 'est_fare', 'eta_estimate_min', 'weather', 'payment_method', 'service_level']


## 12. Final Data Leakage Audit Summary

### Confirmed Leakage

- `actual_trip_time_min`
- `cancel_reason`
- `driver_arrival_min`

### Exclude Until Verified

- `support_ticket_flag`: The failure rate is substantially higher when this flag is 1. Its creation timestamp must be checked.
- `driver_id`: There is no timestamp showing when the driver was assigned. The fact that the driver's join date predates the request does not prove that this driver had already been assigned at prediction time.

### Features Requiring Historical Snapshot Verification

- `loyalty_tier`
- `rating_avg`
- `experience_months`
- `vehicle_type`
- `home_zone` from customer and driver profiles

### Additional Data Quality Notes

- Zone mapping should recognize both `zone_name` and `legacy_name` to avoid losing matches because of naming differences.
- The signup-date and driver-join-date checks found that the available records predate the request timestamps. However, this alone does not prove that all other profile attributes were available at that time.
- No feature can be declared safe without knowing when it was created or confirming that a historical snapshot is available.

**Final decision:** Do not use information generated after `request_ts`. Start by excluding confirmed leakage columns, and keep suspicious columns out until their availability at prediction time is documented. All preprocessing must be fitted on training data only to prevent test-set leakage.
